# 08 - Sync Hyperparameter Sweep

**Role**: Train and evaluate four additional sync-loss hyperparameter variants suggested from `table3_sync_phase_continuation_ablation.md`.

**Additional experiments**
- `phase_trajectory_sync_abs_l018`: trajectory condition + stronger absolute sync (`lambda=0.18`)
- `phase_trajectory_sync_v2_l008`: trajectory condition + softer Sync v2 (`lambda=0.08`, velocity `0.5`, SNR gamma `5`)
- `phase_trajectory_sync_v2_vel025_snr10`: trajectory condition + mild velocity/SNR Sync v2 (`lambda=0.12`, velocity `0.25`, SNR gamma `10`)
- `phase_continuation_sync_soft`: phase-continuation condition + softer sync (`lambda=0.08`, `lr=3e-5`, warm-up `2` epochs)

**Baseline comparison set**
- `phase_trajectory_sync`
- `phase_trajectory_sync_v2`
- `phase_continuation_sync`

**Outputs**
- Four additional checkpoints under `checkpoints/`
- `results/table4_sync_hparam_sweep.md`
- `results/sync_hparam_sweep_results.npz`
- `figures/sync_hparam_*.png`


In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

ensure_package('diffusers')
ensure_package('gymnasium')
ensure_package('mujoco')
ensure_package('tabulate')


## 1. Imports and Data

Load shared artifacts and the frozen estimator. The first three variants use the original phase trajectory condition; only `phase_continuation_sync_soft` uses the four-channel phase-continuation condition.


In [ ]:
import gymnasium as gym
import torch

from pcdp.configs import get_experiment_config, set_global_seed
from pcdp.dataset import build_loaders, load_project_data
from pcdp.evaluation import (
    SYNC_HPARAM_EXTRA_CONFIG_NAMES,
    SYNC_HPARAM_EXTRA_MODEL_KEYS,
    SYNC_HPARAM_SWEEP_CONFIG_NAMES,
    SYNC_HPARAM_SWEEP_MODEL_KEYS,
    build_frequency_sweep_protocol,
    build_frequency_sweep_results_payload,
    load_evaluation_state,
    print_frequency_sweep_summary,
    run_frequency_sweep_evaluation,
    save_eval_results_npz,
    write_frequency_tracking_table_markdown,
)
from pcdp.experiment_plots import (
    plot_evaluation_frequency_comparison,
    plot_frequency_tracking_alignment,
    plot_zone_aggregated_tracking_metrics,
)
from pcdp.experiment_runner import build_model, build_noise_scheduler
from pcdp.frozen_phase_estimator import (
    freeze_phase_estimator,
    load_phase_estimator_checkpoint,
    train_phase_sync_diffusion_policy,
)
from pcdp.paths import CHECKPOINTS_DIR, DATA_DIR, FIGURES_DIR, RESULTS_DIR, ensure_artifact_dirs
from pcdp.training import (
    load_checkpoint,
    save_checkpoint,
    trajectory_phase_cond_fn,
    trajectory_phase_continuation_cond_fn,
)

ensure_artifact_dirs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

data = load_project_data(DATA_DIR)
set_global_seed(data['seed'], deterministic=True)
train_ds, val_ds, train_loader, val_loader = build_loaders(data, batch_size=256, num_workers=2)
print(f"train chunks={len(train_ds):,}, val chunks={len(val_ds):,}")

estimator_ckpt_path = CHECKPOINTS_DIR / 'frozen_phase_estimator_mlp.pt'
if not estimator_ckpt_path.exists():
    raise FileNotFoundError(f'Missing frozen estimator checkpoint: {estimator_ckpt_path}. Run notebook 05 first.')
phase_estimator, _ = load_phase_estimator_checkpoint(estimator_ckpt_path, device=device, use_best=True)
freeze_phase_estimator(phase_estimator)


## 2. Define Four Hyperparameter Variants

`base_config_name` determines the architecture and checkpoint used for initialization. Trajectory variants start from `phase_trajectory_ckpt.pt`; the continuation variant starts from `phase_continuation_ckpt.pt`.


In [ ]:
RUN_SYNC_HPARAM_FINE_TUNE = True
OVERWRITE_SYNC_HPARAM_CHECKPOINTS = False

SYNC_HPARAM_VARIANTS = [
    {
        'config_name': 'phase_trajectory_sync_abs_l018',
        'base_config_name': 'phase_trajectory',
        'base_checkpoint_name': 'phase_trajectory_ckpt.pt',
        'cond_fn': trajectory_phase_cond_fn,
        'description': 'trajectory condition + stronger absolute sync',
        'num_epochs': 15,
        'lr': 5e-5,
        'lambda_phase': 0.18,
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.0,
        'snr_gamma': None,
        'snr_floor': 0.05,
        'phase_warmup_epochs': 1,
    },
    {
        'config_name': 'phase_trajectory_sync_v2_l008',
        'base_config_name': 'phase_trajectory',
        'base_checkpoint_name': 'phase_trajectory_ckpt.pt',
        'cond_fn': trajectory_phase_cond_fn,
        'description': 'trajectory condition + softer velocity/SNR sync',
        'num_epochs': 15,
        'lr': 5e-5,
        'lambda_phase': 0.08,
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.5,
        'snr_gamma': 5.0,
        'snr_floor': 0.05,
        'phase_warmup_epochs': 1,
    },
    {
        'config_name': 'phase_trajectory_sync_v2_vel025_snr10',
        'base_config_name': 'phase_trajectory',
        'base_checkpoint_name': 'phase_trajectory_ckpt.pt',
        'cond_fn': trajectory_phase_cond_fn,
        'description': 'trajectory condition + mild velocity and weaker SNR weighting',
        'num_epochs': 15,
        'lr': 5e-5,
        'lambda_phase': 0.12,
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.25,
        'snr_gamma': 10.0,
        'snr_floor': 0.05,
        'phase_warmup_epochs': 1,
    },
    {
        'config_name': 'phase_continuation_sync_soft',
        'base_config_name': 'phase_continuation',
        'base_checkpoint_name': 'phase_continuation_ckpt.pt',
        'cond_fn': trajectory_phase_continuation_cond_fn,
        'description': 'phase-continuation condition + softer sync fine-tuning',
        'num_epochs': 10,
        'lr': 3e-5,
        'lambda_phase': 0.08,
        'phase_abs_weight': 1.0,
        'phase_velocity_weight': 0.0,
        'snr_gamma': None,
        'snr_floor': 0.05,
        'phase_warmup_epochs': 2,
    },
]

assert tuple(v['config_name'] for v in SYNC_HPARAM_VARIANTS) == SYNC_HPARAM_EXTRA_CONFIG_NAMES


## 3. Train or Load Additional Checkpoints

Each variant is skipped when its checkpoint already exists unless `OVERWRITE_SYNC_HPARAM_CHECKPOINTS=True`.


In [ ]:
sync_hparam_logs = {}

for variant in SYNC_HPARAM_VARIANTS:
    cfg = get_experiment_config(
        variant['config_name'],
        training={
            'num_epochs': variant['num_epochs'],
            'lr': variant['lr'],
            'val_every': 1,
            'val_n_batches': 8,
            'log_every_step': 100,
        },
    )
    ckpt_path = cfg.checkpoint_path(CHECKPOINTS_DIR)
    base_ckpt_path = CHECKPOINTS_DIR / variant['base_checkpoint_name']
    print(f"\n=== {cfg.display_name}: {variant['description']} ===")

    if not base_ckpt_path.exists():
        raise FileNotFoundError(f'Missing base checkpoint: {base_ckpt_path}')
    if ckpt_path.exists() and not OVERWRITE_SYNC_HPARAM_CHECKPOINTS:
        print(f'Skipping existing checkpoint: {ckpt_path}')
        continue
    if not RUN_SYNC_HPARAM_FINE_TUNE:
        raise FileNotFoundError(
            f'Missing checkpoint: {ckpt_path}. Set RUN_SYNC_HPARAM_FINE_TUNE=True to train it.'
        )

    model = build_model(cfg, data, device=device)
    ema = cfg.build_ema(model)
    noise_scheduler, _, _ = build_noise_scheduler(cfg)
    load_checkpoint(base_ckpt_path, model, ema, device=device, use_best_ema=True)

    train_log, val_log, best_ema_state = train_phase_sync_diffusion_policy(
        model,
        ema,
        noise_scheduler,
        phase_estimator,
        train_loader,
        val_loader,
        cond_fn=variant['cond_fn'],
        device=device,
        num_epochs=variant['num_epochs'],
        lr=variant['lr'],
        weight_decay=cfg.training.weight_decay,
        lambda_phase=variant['lambda_phase'],
        phase_abs_weight=variant['phase_abs_weight'],
        phase_velocity_weight=variant['phase_velocity_weight'],
        snr_gamma=variant['snr_gamma'],
        snr_floor=variant['snr_floor'],
        phase_warmup_epochs=variant['phase_warmup_epochs'],
        val_every=cfg.training.val_every,
        val_n_batches=cfg.training.val_n_batches,
        grad_clip=cfg.training.grad_clip,
        log_every_step=cfg.training.log_every_step,
    )
    save_checkpoint(
        ckpt_path,
        model,
        ema,
        train_log,
        val_log,
        best_ema_state,
        config={
            **cfg.to_dict(),
            'sync_hparam_variant': {
                key: value for key, value in variant.items() if key != 'cond_fn'
            },
            'phase_estimator_checkpoint': str(estimator_ckpt_path),
            'source_checkpoint': str(base_ckpt_path),
        },
    )
    sync_hparam_logs[variant['config_name']] = {
        'train_log': train_log,
        'val_log': val_log,
        'checkpoint': ckpt_path,
    }


## 4. Frequency-Sweep Evaluation

The default comparison includes the three strongest table-3 baselines plus the four new hyperparameter variants. Reduce `EVAL_CONFIG_NAMES` if runtime is too high.


In [ ]:
RUN_HPARAM_EVALUATION = True
N_SEEDS_SWEEP = 10
MAX_STEPS = 1000
DT = 0.05

EVAL_CONFIG_NAMES = SYNC_HPARAM_SWEEP_CONFIG_NAMES
EVAL_MODEL_KEYS = SYNC_HPARAM_SWEEP_MODEL_KEYS

if RUN_HPARAM_EVALUATION:
    freq_protocol = build_frequency_sweep_protocol(data, n_in_dist=3, ood_iqr_scale=1.5)
    state = load_evaluation_state(
        data,
        device=device,
        checkpoints_dir=CHECKPOINTS_DIR,
        config_names=EVAL_CONFIG_NAMES,
    )

    env = gym.make('Ant-v5')
    try:
        hparam_freq_results = run_frequency_sweep_evaluation(
            state,
            freq_protocol,
            env=env,
            data=data,
            device=device,
            model_keys=EVAL_MODEL_KEYS,
            n_seeds=N_SEEDS_SWEEP,
            max_steps=MAX_STEPS,
            dt=DT,
        )
    finally:
        env.close()

    print_frequency_sweep_summary(
        data,
        freq_protocol,
        hparam_freq_results,
        model_keys=EVAL_MODEL_KEYS,
    )

    table_path = write_frequency_tracking_table_markdown(
        freq_protocol,
        hparam_freq_results,
        RESULTS_DIR / 'table4_sync_hparam_sweep.md',
        model_keys=EVAL_MODEL_KEYS,
        interval='ci95',
    )

    payload = build_frequency_sweep_results_payload(
        data,
        freq_protocol,
        hparam_freq_results,
        model_keys=EVAL_MODEL_KEYS,
        n_seeds_sweep=N_SEEDS_SWEEP,
    )
    npz_path = save_eval_results_npz(
        payload,
        RESULTS_DIR / 'sync_hparam_sweep_results.npz',
    )
    print(table_path)
    print(npz_path)
else:
    print('Set RUN_HPARAM_EVALUATION=True to run rollouts.')


## 5. Hyperparameter Sweep Figures

These figures mirror the ablation notebook and use only the selected `EVAL_MODEL_KEYS`.


In [ ]:
if RUN_HPARAM_EVALUATION:
    plot_evaluation_frequency_comparison(
        {},
        hparam_freq_results,
        data,
        FIGURES_DIR / 'sync_hparam_reward_per_step_vs_freq.png',
        n_seeds_sweep=N_SEEDS_SWEEP,
        phase_model_keys=EVAL_MODEL_KEYS,
        state=state,
    )
    plot_frequency_tracking_alignment(
        hparam_freq_results,
        data,
        FIGURES_DIR / 'sync_hparam_target_vs_measured_freq.png',
        n_seeds_sweep=N_SEEDS_SWEEP,
        phase_model_keys=EVAL_MODEL_KEYS,
        state=state,
        interval='ci95',
        periodic_collapse_note=False,
    )
    plot_zone_aggregated_tracking_metrics(
        freq_protocol,
        hparam_freq_results,
        FIGURES_DIR / 'sync_hparam_zone_tracking_metrics.png',
        phase_model_keys=EVAL_MODEL_KEYS,
        state=state,
        interval='ci95',
    )
